# NeuroGolf Submission Folder Score

指定したfolder内の`taskXXX.onnx`を読み込み、正解判定なしでNeuroGolf Metricのcost/scoreだけを計算します。

In [5]:
from pathlib import Path
import json
import math
import tempfile

import numpy as np
import onnx
import onnxruntime


repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

DATA_DIR = repo_root / "neurogolf-2026"

# Change this path to score another folder, e.g. repo_root / "solution" / "6405.61".
SUBMISSION_DIR = repo_root / "solution" / "6405.61"

SUBMISSION_DIR

PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6405.61')

In [6]:
FILESIZE_LIMIT_IN_BYTES = 1.44 * 1024 * 1024
EXCLUDED_OP_TYPES = {"LOOP", "SCAN", "NONZERO", "UNIQUE", "SCRIPT", "FUNCTION", "COMPRESS"}
def fallback_input():
    x = np.zeros((1, 10, 30, 30), dtype=np.float32)
    x[0, 0, :, :] = 1.0
    return x


def grid_to_input_tensor(grid):
    x = np.zeros((1, 10, 30, 30), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            x[0, color, r, c] = 1.0
    return x


def profiling_input_for_task(task_name):
    task_path = DATA_DIR / f"{task_name}.json"
    if not task_path.exists():
        return fallback_input()
    with open(task_path) as f:
        task = json.load(f)
    return grid_to_input_tensor(task["train"][0]["input"])


def calculate_params(model):
    params = 0
    for init in model.graph.initializer:
        if any(d <= 0 for d in init.dims):
            return None
        params += math.prod(init.dims)
    for sparse_init in model.graph.sparse_initializer:
        if any(d <= 0 for d in sparse_init.values.dims):
            return None
        params += math.prod(sparse_init.values.dims)
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                if any(d <= 0 for d in attr.t.dims):
                    return None
                params += math.prod(attr.t.dims)
            elif attr.name == "sparse_value":
                if any(d <= 0 for d in attr.sparse_tensor.values.dims):
                    return None
                params += math.prod(attr.sparse_tensor.values.dims)
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return params


def sanitize_model(model):
    for node in model.graph.node:
        if node.output:
            node.name = node.output[0]
            if "kernel_time" in node.output[0]:
                return None

    name_map = {}
    counter = 0

    def get_safe_name(old_name):
        nonlocal counter
        if not old_name or old_name in {"input", "output"}:
            return old_name
        if old_name not in name_map:
            name_map[old_name] = f"safe_name_{counter}"
            counter += 1
        return name_map[old_name]

    for inp in model.graph.input:
        inp.name = get_safe_name(inp.name)
    for init in model.graph.initializer:
        init.name = get_safe_name(init.name)
    for node in model.graph.node:
        for i in range(len(node.input)):
            node.input[i] = get_safe_name(node.input[i])
        for i in range(len(node.output)):
            node.output[i] = get_safe_name(node.output[i])
        if node.output and node.output[0]:
            node.name = node.output[0]
    for out in model.graph.output:
        out.name = get_safe_name(out.name)
    for vi in model.graph.value_info:
        vi.name = get_safe_name(vi.name)
    for node in model.graph.node:
        if node.output:
            node.name = node.output[0]
    return model


def calculate_memory(model, trace_path):
    onnx.checker.check_model(model, full_check=True)
    graph = onnx.shape_inference.infer_shapes(model, strict_mode=True).graph
    if len(graph.input) > 1 or len(graph.output) > 1:
        return None

    init_names = {init.name for init in graph.initializer}
    init_names.update(init.name for init in graph.sparse_initializer)
    io_names = {t.name for t in list(graph.input) + list(graph.output)}
    if io_names.intersection(init_names):
        return None
    if model.functions:
        return None
    for opset in model.opset_import:
        if opset.domain not in {"", "ai.onnx"}:
            return None

    node_outputs = {}
    tensor_names = set()
    for node in graph.node:
        for attr in node.attribute:
            if attr.type in [onnx.AttributeProto.GRAPH, onnx.AttributeProto.GRAPHS]:
                return None
        node_outputs[node.name] = list(node.output)
        for output_name in node.output:
            if output_name:
                tensor_names.add(output_name)

    tensor_memory = {}
    tensor_dtypes = {}
    tensor_map = {t.name: t for t in list(graph.input) + list(graph.value_info) + list(graph.output)}
    tensor_names.update(tensor_map.keys())
    for tensor_name in tensor_names:
        item = tensor_map.get(tensor_name)
        if not item:
            return None
        if item.type.HasField("sequence_type"):
            return None
        if not item.type.HasField("tensor_type"):
            continue
        tensor_type = item.type.tensor_type
        if not tensor_type.HasField("shape"):
            return None
        num_elements = 1
        for dim in tensor_type.shape.dim:
            if dim.HasField("dim_param") or not dim.HasField("dim_value") or dim.dim_value <= 0:
                return None
            num_elements *= dim.dim_value
        if tensor_name in ["input", "output"]:
            continue
        np_dtype = onnx.helper.tensor_dtype_to_np_dtype(tensor_type.elem_type)
        tensor_memory[tensor_name] = num_elements * np.dtype(np_dtype).itemsize
        tensor_dtypes[tensor_name] = np_dtype

    seen = set()
    for item in list(graph.input) + list(graph.value_info) + list(graph.output):
        if item.name in seen:
            return None
        seen.add(item.name)
    for node in graph.node:
        for output_name in node.output:
            if output_name and output_name != "output":
                item = tensor_map.get(output_name)
                if item is None or not item.type.HasField("tensor_type"):
                    return None

    with open(trace_path) as f:
        trace_data = json.load(f)
    for event in trace_data:
        if event.get("cat") != "Node" or "args" not in event:
            continue
        if "output_type_shape" not in event["args"]:
            continue
        node_name = event.get("name").replace("_kernel_time", "")
        if node_name not in node_outputs:
            continue
        for i, shape_dict in enumerate(event["args"]["output_type_shape"]):
            if i >= len(node_outputs[node_name]):
                continue
            output_name = node_outputs[node_name][i]
            if output_name not in tensor_dtypes:
                continue
            itemsize = np.dtype(tensor_dtypes[output_name]).itemsize
            mem = itemsize * sum(math.prod(dims) for dims in shape_dict.values())
            tensor_memory[output_name] = max(tensor_memory[output_name], mem)
    return sum(tensor_memory.values())


def calculate_static_memory(model):
    onnx.checker.check_model(model, full_check=True)
    graph = onnx.shape_inference.infer_shapes(model, strict_mode=True).graph
    if len(graph.input) > 1 or len(graph.output) > 1:
        return None

    init_names = {init.name for init in graph.initializer}
    init_names.update(init.name for init in graph.sparse_initializer)
    io_names = {t.name for t in list(graph.input) + list(graph.output)}
    if io_names.intersection(init_names):
        return None
    if model.functions:
        return None
    for opset in model.opset_import:
        if opset.domain not in {"", "ai.onnx"}:
            return None

    tensor_names = set()
    for node in graph.node:
        for attr in node.attribute:
            if attr.type in [onnx.AttributeProto.GRAPH, onnx.AttributeProto.GRAPHS]:
                return None
        for output_name in node.output:
            if output_name:
                tensor_names.add(output_name)

    tensor_map = {t.name: t for t in list(graph.input) + list(graph.value_info) + list(graph.output)}
    tensor_names.update(tensor_map.keys())
    total_memory = 0
    for tensor_name in tensor_names:
        item = tensor_map.get(tensor_name)
        if not item:
            return None
        if item.type.HasField("sequence_type"):
            return None
        if not item.type.HasField("tensor_type"):
            continue
        tensor_type = item.type.tensor_type
        if not tensor_type.HasField("shape"):
            return None
        num_elements = 1
        for dim in tensor_type.shape.dim:
            if dim.HasField("dim_param") or not dim.HasField("dim_value") or dim.dim_value <= 0:
                return None
            num_elements *= dim.dim_value
        if tensor_name in ["input", "output"]:
            continue
        np_dtype = onnx.helper.tensor_dtype_to_np_dtype(tensor_type.elem_type)
        total_memory += num_elements * np.dtype(np_dtype).itemsize

    seen = set()
    for item in list(graph.input) + list(graph.value_info) + list(graph.output):
        if item.name in seen:
            return None
        seen.add(item.name)
    for node in graph.node:
        for output_name in node.output:
            if output_name and output_name != "output":
                item = tensor_map.get(output_name)
                if item is None or not item.type.HasField("tensor_type"):
                    return None
    return total_memory


def finalize_score_result(result, memory, params, status, error=""):
    if memory is None or params is None or memory < 0 or params < 0:
        result.update(status="error", error="cost could not be measured")
        return result
    cost = max(1.0, float(memory + params))
    score = max(1.0, 25.0 - math.log(cost))
    result.update(status=status, memory=memory, params=params, cost=cost, score=score, error=error)
    return result


def score_model_file(path):
    path = Path(path)
    result = {"task": path.stem, "file": str(path), "filesize": path.stat().st_size}
    if result["filesize"] > FILESIZE_LIMIT_IN_BYTES:
        result.update(status="error", error="filesize limit exceeded")
        return result

    try:
        model = sanitize_model(onnx.load(path))
        if model is None:
            result.update(status="error", error="model sanitization failed")
            return result
        for node in model.graph.node:
            op_type = node.op_type.upper()
            if op_type in EXCLUDED_OP_TYPES or "Sequence" in node.op_type:
                result.update(status="error", error=f"excluded op: {node.op_type}")
                return result

        with tempfile.TemporaryDirectory() as tmpdir:
            options = onnxruntime.SessionOptions()
            options.enable_profiling = True
            options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
            options.profile_file_prefix = str(Path(tmpdir) / path.stem)
            session = onnxruntime.InferenceSession(model.SerializeToString(), options)
            session.run(["output"], {"input": profiling_input_for_task(path.stem)})
            trace_path = session.end_profiling()

            memory = calculate_memory(model, trace_path)
            params = calculate_params(model)
        return finalize_score_result(result, memory, params, "ok")
    except Exception as exc:
        try:
            model = sanitize_model(onnx.load(path))
            if model is None:
                result.update(status="error", error="model sanitization failed")
                return result
            memory = calculate_static_memory(model)
            params = calculate_params(model)
            return finalize_score_result(result, memory, params, "ok_static", repr(exc))
        except Exception as static_exc:
            result.update(status="error", error=f"runtime={repr(exc)}; static={repr(static_exc)}")
            return result


def score_submission_folder(folder):
    folder = Path(folder)
    rows = [score_model_file(path) for path in sorted(folder.glob("task*.onnx"))]
    ok_rows = [row for row in rows if row["status"] in {"ok", "ok_static"}]
    return {
        "folder": str(folder),
        "num_files": len(rows),
        "num_ok": len(ok_rows),
        "num_errors": len(rows) - len(ok_rows),
        "total_score": sum(row["score"] for row in ok_rows),
        "rows": rows,
    }

In [7]:
summary = score_submission_folder(SUBMISSION_DIR)
print(f"folder      : {summary['folder']}")
print(f"files       : {summary['num_files']}")
print(f"ok          : {summary['num_ok']}")
print(f"errors      : {summary['num_errors']}")
print(f"total score : {summary['total_score']:.6f}")

2026-06-16 21:57:46.794 python[75895:17836756] 2026-06-16 21:57:46.791489 [E:onnxruntime:, inference_session.cc:3540 EndProfiling] Could not write a profile because no model was loaded.


folder      : /Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6405.61
files       : 400
ok          : 400
errors      : 0
total score : 6405.611247


In [8]:
try:
    import pandas as pd

    df = pd.DataFrame(summary["rows"])
    display(df.sort_values("task").reset_index(drop=True))
except ImportError:
    summary["rows"][:10]

,task,file,filesize,status,memory,params,cost,score,error
0,task001,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,1172,ok,2666,24,2690.0,17.102704,
1,task002,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,9145,ok,86400,46,86446.0,13.632725,
2,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,5000,ok,1172,18,1190.0,17.918291,
3,task004,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,3106,ok,40960,1045,42005.0,14.354456,
4,task005,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,79215,ok,61780,1911,63691.0,13.938201,
...,...,...,...,...,...,...,...,...,...
395,task396,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,8843,ok,31785,511,32296.0,14.617301,
396,task397,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,6674,ok,19741,87,19828.0,15.105150,
397,task398,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,8089,ok_static,33140,754,33894.0,14.569007,"NotImplemented(""[ONNXRuntimeError] : 9 : NOT_I..."
398,task399,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,1744,ok,912,272,1184.0,17.923346,


Note: このnotebookは正解/不正解の検証をしません。ONNXが正解すると仮定して、提出folder内の各ファイルのcostからscoreだけを推定します。`status == "ok_static"`は、ローカルONNX Runtimeでprofiling実行できなかったため静的shape推定のmemoryでfallbackした行です。